In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt


In [ ]:
def variance_reduction_factor(p, a):

    return stats.chi2.cdf(a, p + 2) / stats.chi2.cdf(a, p)


def simulate_L(p, a, N, rng):

    g = rng.standard_normal((N, p))
    u1 = g[:, 0] / np.sqrt((g ** 2).sum(axis=1))
    Fa = stats.chi2.cdf(a, p)
    U = rng.uniform(0.0, Fa, size=N)
    W = stats.chi2.ppf(U, p)
    return np.sqrt(W) * u1


def quantile_curves(p, pa, alpha, R2_grid, N=2_000_000, seed=0):


    rng = np.random.default_rng(seed)
    a = stats.chi2.ppf(pa, p)
    vp = variance_reduction_factor(p, a)
    eps0 = rng.standard_normal(N)
    L = simulate_L(p, a, N, rng)
    z = stats.norm.ppf(1 - alpha / 2)
    prob = 1 - alpha / 2
    rows = []
    for R2 in R2_grid:
        Q = np.sqrt(1 - R2) * eps0 + np.sqrt(R2) * L
        nu = np.quantile(Q, prob)
        q = nu / np.sqrt(1 - (1 - vp) * R2)
        rows.append((R2, nu, q))
    df = pd.DataFrame(rows, columns=["R2", "nu", "q"])
    df["z"] = z
    df.attrs.update(p=p, pa=pa, alpha=alpha, a=a, vp=vp, z=z, N=N)
    return df


def worst_case(df):

    i_nu, i_q = df["nu"].idxmax(), df["q"].idxmax()
    return dict(nu_bar=df.loc[i_nu, "nu"], R2_at_nu_bar=df.loc[i_nu, "R2"],
                q_bar=df.loc[i_q, "q"],   R2_at_q_bar=df.loc[i_q, "R2"],
                z=df.attrs["z"], vp=df.attrs["vp"], a=df.attrs["a"])


In [ ]:
rng = np.random.default_rng(7)
for p, pa in [(2, 0.01), (10, 0.01), (20, 0.1)]:
    a = stats.chi2.ppf(pa, p)
    L = simulate_L(p, a, 2_000_000, rng)
    print(f"p={p:2d} pa={pa}: Var(L) sim={L.var():.5f}  theory v_p={variance_reduction_factor(p,a):.5f}"
          f"  mean(L)={L.mean():+.5f}")

df0 = quantile_curves(5, 0.01, 0.05, np.array([0.0]), N=2_000_000, seed=3)
print(f"R2=0: nu={df0.nu[0]:.5f}  q={df0.q[0]:.5f}  z={df0.attrs['z']:.5f}  (all equal)")


In [ ]:
p, pa, alpha = 10, 0.01, 0.05

R2_fine = np.linspace(0, 1, 201)
df = quantile_curves(p, pa, alpha, R2_fine, N=1_000_000, seed=0)


show = df[np.isin(df["R2"].round(2), [0.0, 0.25, 0.5, 0.75, 0.9, 1.0])].copy()
show = show.rename(columns={"nu": "nu(R2)", "q": "q(R2)", "z": "z_(1-a/2)"})
print("POINTWISE quantiles  (p=%d, pa=%g, alpha=%g):" % (p, pa, alpha))
print(show.round(4).to_string(index=False))


wc = worst_case(df)
print("\nWORST-CASE (eq 17):")
print(f"  nu_bar = sup_R2 nu = {wc['nu_bar']:.4f}   (attained at R2={wc['R2_at_nu_bar']:.3f})")
print(f"  q_bar  = sup_R2 q  = {wc['q_bar']:.4f}   (attained at R2={wc['R2_at_q_bar']:.3f})")
print(f"  z_(1-a/2)          = {wc['z']:.4f}")
print(f"  nu_bar/z = {wc['nu_bar']/wc['z']:.4f}    q_bar/z = {wc['q_bar']/wc['z']:.4f}")


In [ ]:
def worst_case_with_se(p, pa, alpha, R2_grid, N, reps, seed0):
    nb_, qb_, rq_ = [], [], []
    for r in range(reps):
        wc = worst_case(quantile_curves(p, pa, alpha, R2_grid, N=N, seed=seed0 + r))
        nb_.append(wc["nu_bar"]); qb_.append(wc["q_bar"]); rq_.append(wc["R2_at_q_bar"])
    nb_, qb_ = np.array(nb_), np.array(qb_)
    a = stats.chi2.ppf(pa, p); vp = variance_reduction_factor(p, a)
    z = stats.norm.ppf(1 - alpha / 2)
    return dict(alpha=alpha, p=p, pa=pa, a=round(a, 3), v_p=round(vp, 4), z=round(z, 5),
                nu_bar=nb_.mean(), nu_se=nb_.std(ddof=1)/np.sqrt(reps),
                q_bar=qb_.mean(),  q_se=qb_.std(ddof=1)/np.sqrt(reps),
                R2_at_q_bar=float(np.median(rq_)),
                nu_bar_over_z=nb_.mean()/z, q_bar_over_z=qb_.mean()/z)


def summary_table(p_list, pa_list, alpha_list, R2_grid, N, reps, seed0=100):
    recs = [worst_case_with_se(p, pa, alpha, R2_grid, N, reps, seed0)
            for alpha in alpha_list for p in p_list for pa in pa_list]
    tab = pd.DataFrame(recs)
    for col in ["nu_bar","nu_se","q_bar","q_se","nu_bar_over_z","q_bar_over_z"]:
        tab[col] = tab[col].round(5)
    return tab


In [ ]:
R2_grid    = np.linspace(0.0, 1.0, 51)
p_list     = [1, 2, 5, 10, 20]
pa_list    = [0.001, 0.01, 0.05, 0.1]
alpha_list = [0.05, 0.10]


tab = summary_table(p_list, pa_list, alpha_list, R2_grid, N=400_000, reps=4)
pd.set_option("display.width", 220, "display.max_columns", 30)
tab


In [ ]:
tab.to_csv("rerand_quantiles_summary.csv", index=False)
print("saved rerand_quantiles_summary.csv")


In [ ]:
R2 = np.linspace(0, 1, 101)
alpha = 0.05
z = stats.norm.ppf(1 - alpha / 2)
pa_rows = [0.01, 0.05]
ps = [2, 5, 10, 20]

plt.rcParams["savefig.format"] = "pdf"
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True)
for i, pa in enumerate(pa_rows):
    for p in ps:
        dfc = quantile_curves(p, pa, alpha, R2, N=1_000_000, seed=11)
        axes[i, 0].plot(R2, dfc["nu"], label=f"p={p}")
        axes[i, 1].plot(R2, dfc["q"],  label=f"p={p}")
    for j, (ttl, sym) in enumerate(zip(
            [r"$\nu_{1-\alpha/2}(R^2;p,a)$", r"$q_{1-\alpha/2}(R^2;p,a)$"],
            [r"$\nu$", r"$q$"])):
        ax = axes[i, j]
        ax.axhline(z, ls="--", color="k", lw=1, label=r"$z_{1-\alpha/2}$")
        ax.set_ylabel(sym); ax.grid(alpha=.3)
        ax.set_title(rf"{ttl}   ($p_a={pa}$)")
        ax.legend(fontsize=8)
        if i == 1:
            ax.set_xlabel(r"$R^2$")
fig.suptitle(r"quantiles and gaussian quantile ($\alpha=0.05$)")
fig.tight_layout()
fig.savefig("quantile_curves.pdf", bbox_inches="tight")
plt.show()
